In [1]:
from ultralytics import YOLO
from pytubefix import YouTube
from pathlib import Path
import cv2
import os
import shutil
import torch
import glob

print(torch.cuda.is_available())
print(torch.cuda.current_device())
print(torch.cuda.get_device_name(0))

True
0
NVIDIA GeForce RTX 4050 Laptop GPU


In [ ]:
### FOR TRAINING ONLY

model = YOLO("yolo11m.pt")

model.train(
    data="datasets/cctv/data.yaml",
    epochs=100,
    device=0,
    amp=True,
    imgsz=512, # Keept at 512. 640 is too slow
    batch=16, # Keep at 8 or lower
    workers=4, # Keep at 4 or lower
    plots=True # False just in case
)

In [ ]:
### Main Code Area

detect_model = YOLO("models/11m.pt")
pose_model = YOLO("yolo11m-pose.pt")


min_conf = 0.5
input_directory = Path("video/input")
output_directory = Path("video/output")
output_directory.mkdir(parents=True, exist_ok=True)

output_video_count = 0
output_path = "video/output"
fourcc = cv2.VideoWriter_fourcc(*"mp4v")
fps = 30

first = True
writer = None
frame_count = 0

detect_results = detect_model.track(
    source="video/input/test_video_3.mp4",
    device=0,
    half=True,
    stream=True,
    tracker="botsort.yaml"
)

input_videos = []
input_videos.extend(input_directory.glob("*.mp4"))

for video in input_videos:
    print(f"Processing: {video.name}")

    detect_results = detect_model.track(
        source=str(video),
        device=0,
        half=True,
        stream=True,
        tracker="botsort.yaml"
    )

    writer = None

    for result in detect_results:

        frame = result.orig_img.copy()
        boxes = result.boxes
        if boxes is None or len(boxes) == 0:
            continue

        track_ids = (
            boxes.id.int().cpu().tolist()
            if boxes.id is not None
            else [None] * len(boxes)
        )

        if writer is None:
            height, width = frame.shape[:2]

            output_file = output_directory / f"{video.stem}.mp4"

            writer = cv2.VideoWriter(
                str(output_file),
                fourcc,
                30,
                (width, height)
            )
        
        for box, cls_id, conf, track_id in zip(
            boxes.xyxy,
            boxes.cls,
            boxes.conf,
            track_ids
        ):
            # Frame Drawing
            x1, y1, x2, y2 = map(int, box)
            rgb = (0, 0, 255)
            thickness = 2
            font_scale = 0.6
            cls_name = detect_model.names[int(cls_id)]
            label = f"ID:{track_id} CLS:{cls_name} CONF:{conf:.2f}"

            cv2.rectangle(
                frame,
                (x1, y1),
                (x2, y2),
                rgb,
                thickness
            )
            cv2.putText(
                frame,
                label,
                (x1, y1 - 5),
                cv2.FONT_HERSHEY_SIMPLEX,
                font_scale,
                rgb,
                thickness
            )

            # Guard Clauses
            not_person = result.names[int(cls_id)] != "person"
            not_min_conf = conf < min_conf
            if not_person or not_min_conf:
                continue
            not_valid_crop = x2 <= x1 or y2 <= y1
            if not_valid_crop:
                continue
            crop = frame[y1:y2, x1:x2]
            if crop.size == 0:
                continue

            # Pose Estimation
            pose_results = pose_model(
                crop,
                verbose=False
            )
            pose_result = pose_results[0]
            annotated_crop = pose_result.plot(
                boxes=False,
                labels=False
            )
            resized = cv2.resize(
                annotated_crop,
                (x2 - x1, y2 - y1)
            )
            frame[y1:y2, x1:x2] = resized

        writer.write(frame)

    if writer is not None:
        writer.release()

    print(f"Finished: {video.name}")

print(f"All Done")